In [2]:
import importlib.metadata as importlib_metadata
import inspect
import logging
import os
import openai
import huggingface_hub
import torch

os.environ["CC"] = "/usr/bin/gcc"
os.environ["CXX"] = "/usr/bin/g++"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def import_vllm_or_raise():
    try:
        from vllm import LLM, SamplingParams
        return LLM, SamplingParams
    except ImportError as error:
        torch_version = importlib_metadata.version("torch")
        try:
            vllm_version = importlib_metadata.version("vllm")
        except importlib_metadata.PackageNotFoundError:
            vllm_version = "not installed"

        raise RuntimeError(
            "vLLM import failed because the installed vLLM extension is not ABI-compatible with the current PyTorch build.\n"
            f"- torch: {torch_version}\n"
            f"- torch CUDA: {torch.version.cuda}\n"
            f"- vllm: {vllm_version}\n"
            f"- original error: {error}\n\n"
            "This environment currently has torch 2.10.0+cu126, and older vLLM wheels such as 0.2.5 are not compatible with that ABI. "
            "Reinstall a vLLM build that matches your current torch/CUDA stack, or recreate the environment with a compatible torch+vllm pair."
        ) from error

LLM, SamplingParams = import_vllm_or_raise()

def maybe_hf_login() -> None:
    token = (
        os.getenv("HF_TOKEN")
        or os.getenv("HUGGINGFACE_TOKEN")
        or os.getenv("HUGGINGFACE_HUB_TOKEN")
    )
    if not token:
        return

    if not str(token).startswith("hf_"):
        logging.warning("Ignoring Hugging Face token from env because it does not look like an access token.")
        return

    login_kwargs = {
        "token": token,
        "add_to_git_credential": False,
    }
    if "skip_if_logged_in" in inspect.signature(huggingface_hub.login).parameters:
        login_kwargs["skip_if_logged_in"] = True

    huggingface_hub.login(**login_kwargs)

def ensure_cuda_available() -> None:
    cuda_available = torch.cuda.is_available()
    device_count = torch.cuda.device_count()
    if cuda_available and device_count > 0:
        return

    raise RuntimeError(
        "vLLM requires an NVIDIA GPU visible to this notebook kernel, but no CUDA device is currently available.\n"
        f"- torch.cuda.is_available(): {cuda_available}\n"
        f"- torch.cuda.device_count(): {device_count}\n\n"
        "Check that you selected the vLLM kernel on the GPU machine, the NVIDIA driver is installed, and CUDA_VISIBLE_DEVICES is not empty."
    )

maybe_hf_login()
ensure_cuda_available()



llm = LLM(
    model="daje/meta-llama3.1-8B-qna-koalpaca-v1.1",
    dtype="float16",
    tensor_parallel_size=2,
    gpu_memory_utilization=0.88,
    max_model_len=512,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    attention_config={"backend": "TRITON_ATTN"},
)




INFO 03-30 11:52:08 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': 'daje/meta-llama3.1-8B-qna-koalpaca-v1.1'}
INFO 03-30 11:52:09 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 03-30 11:52:09 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-30 11:52:09 [model.py:1582] Using max model len 512
INFO 03-30 11:52:09 [scheduler.py:2

(Worker pid=3854876) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=3854876) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=3854877) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=3854877) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(Worker pid=3854876) INFO 03-30 11:52:20 [pynccl.py:111] vLLM is using nccl==2.27.5
NCCL version 2.27.5+cuda12.9
(Worker pid=3854876) WARNING 03-30 11:52:21 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=3854877) WARNING 03-30 11:52:21 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=3854876) INFO 03-30 11:52:21 [parallel_state.py:1717] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(Worker pid=3854877) INFO 03-30 11:52:21 [parallel_state.py:1717] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP rank N/A, EPLB rank N/A
(Worker_TP0 pid=3854876) INFO 03-30 11:52:21 [gpu_model_runner.py:4481] Starting to load model daje/meta-llama3.1-8B-qna-koalpaca-v1.1...
(Worker_TP0 pid=3854876) INFO 03-30 11:52:22 [cuda.py:257] Using AttentionBackendEnum.TRITON_ATTN

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:07,  2.38s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:04<00:04,  2.33s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:07<00:02,  2.48s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  1.83s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  2.04s/it]
(Worker_TP0 pid=3854876) 


(Worker_TP0 pid=3854876) INFO 03-30 11:52:32 [default_loader.py:384] Loading weights took 8.18 seconds
(Worker_TP0 pid=3854876) INFO 03-30 11:52:32 [gpu_model_runner.py:4566] Model loading took 7.51 GiB memory and 10.043534 seconds
(Worker_TP0 pid=3854876) INFO 03-30 11:52:38 [gpu_worker.py:456] Available KV cache memory: 1.5 GiB
(EngineCore pid=3854769) INFO 03-30 11:52:38 [kv_cache_utils.py:1316] GPU KV cache size: 24,592 tokens
(EngineCore pid=3854769) INFO 03-30 11:52:38 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 48.03x
(EngineCore pid=3854769) INFO 03-30 11:52:39 [core.py:281] init engine (profile, create kv cache, warmup model) took 6.40 seconds
(EngineCore pid=3854769) WARNING 03-30 11:52:42 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
(EngineCore pid=3854769) INFO 03-30 11:52:42 [vllm.py:754] Asynchronous scheduling is enabled.
(EngineCore pid=3854769) WARNING 03-30 11:52:42 [vllm.py:788] Enfor